# Session Time 이상치 제거 분석

실무에서 수행한 Session Time 분석 코드를 기반으로 작성한 공개용 Notebook입니다.

- 실제 회사 데이터 및 내부 식별 정보 제외
- 공개용 합성 데이터 사용
- 원본 분석 순서 및 코드 구조 최대한 유지
- 내부 컬럼명과 콘텐츠 정보만 일반화


In [ ]:
### 라이브러리 호출

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from pathlib import Path


### 공개용 데이터 경로 설정

path = Path("../sample_data/")


### 공개용 합성 데이터 호출

sample = pd.read_csv(path / "sample_sessiontime.csv")


### 주차별 로그 통합 구조 재현

all_result = []

for week in sorted(sample["week"].unique()):
    df_week = sample[sample["week"] == week][
        [
            "week",
            "date",
            "content_id",
            "set_id",
            "user_id",
            "scroll_segment",
            "session_time",
            "content_length",
        ]
    ].copy()

    all_result.append(df_week)

final = pd.concat(all_result, ignore_index=True)

final


In [ ]:
### 콘텐츠 메타데이터 구성

c_list = (
    sample[
        [
            "week",
            "content_id",
            "section",
            "slot",
            "curation_type",
            "app_type",
            "content_type",
            "content_name",
            "content_group",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "week": "Week",
            "content_id": "ContentKey",
        }
    )
)


### 결합 키 정리

c_list["Week"] = c_list["Week"].astype(str).str.strip()
c_list["ContentKey"] = c_list["ContentKey"].astype(str).str.strip()

final["week"] = final["week"].astype(str).str.strip()
final["content_id"] = final["content_id"].astype(str).str.strip()


### 결측치 처리 및 확인

final["session_time"] = final["session_time"].fillna(0)

final.isnull().sum()


In [ ]:
### 데이터 크기 확인

final.shape


In [ ]:
### 로그 및 콘텐츠 메타데이터 결합

final_df = final.copy()

final_df = pd.merge(
    final_df,
    c_list,
    right_on=["Week", "ContentKey"],
    left_on=["week", "content_id"],
    how="left",
)

final_df.shape


In [ ]:
### 분석 컬럼 정리

final_df = final_df[
    [
        "Week",
        "date",
        "section",
        "slot",
        "curation_type",
        "app_type",
        "content_type",
        "content_name",
        "content_group",
        "ContentKey",
        "set_id",
        "user_id",
        "scroll_segment",
        "session_time",
        "content_length",
    ]
]


### 원본 분석 구조에 따른 정수형 변환

final_df["session_time"] = final_df["session_time"].astype(int)

final_df.head()


In [ ]:
### 스크롤 뎁스별 Session Time 분포 확인

plt.figure(figsize=(12, 6))

sns.kdeplot(
    data=final_df[final_df["session_time"] <= 500],
    x="session_time",
    hue="scroll_segment",
    fill=True,
    common_norm=False,
)

plt.xlim(0, 300)
plt.show()


In [ ]:
### Session Time 구간별 비중 확인

total = len(final_df)

count_0_20 = (
    (final_df["session_time"] >= 0)
    & (final_df["session_time"] <= 20)
).sum()

count_21_300 = (
    (final_df["session_time"] >= 21)
    & (final_df["session_time"] <= 300)
).sum()

count_over300 = (
    final_df["session_time"] > 300
).sum()


result = pd.DataFrame(
    {
        "bins": [
            "Total",
            "0s~20s",
            "21s~300s",
            "over300s",
        ],
        "count": [
            total,
            count_0_20,
            count_21_300,
            count_over300,
        ],
    }
)

result["ratio"] = result["count"] / total

result


In [ ]:
### 300초 초과 Session 제외

final_df = final_df[
    final_df["session_time"] <= 300
].copy()


### 스크롤 뎁스 구간명 정리

final_df["scroll_segment"] = final_df["scroll_segment"].replace(
    "OUT",
    "즉시 이탈",
)

final_df["scroll_segment"] = final_df["scroll_segment"].replace(
    "0-25%",
    "0% ~ 25%",
)

final_df["scroll_segment"] = final_df["scroll_segment"].replace(
    "25-50%",
    "25% ~ 50%",
)

final_df["scroll_segment"] = final_df["scroll_segment"].replace(
    "50-75%",
    "50% ~ 75%",
)

final_df["scroll_segment"] = final_df["scroll_segment"].replace(
    "75-100%",
    "75% ~ 100%",
)

final_df["scroll_segment"] = final_df["scroll_segment"].replace(
    "COMPLETE",
    "완독",
)


In [ ]:
### 콘텐츠 유형별 Session Time 집계

content_summary = (
    final_df.groupby(
        [
            "curation_type",
            "app_type",
            "content_group",
            "content_name",
            "scroll_segment",
        ]
    )[["session_time"]]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
    .reset_index()
)

content_summary.head(20)


In [ ]:
### 큐레이션 유형 및 앱 유형별 Session Time 집계

type_summary = (
    final_df.groupby(
        [
            "curation_type",
            "app_type",
            "scroll_segment",
        ]
    )[["session_time"]]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
    .reset_index()
)

type_summary.head(20)


In [ ]:
### 주차 및 슬롯별 Session Time 집계

weekly_slot_summary = (
    final_df.groupby(
        [
            "Week",
            "slot",
            "curation_type",
            "content_name",
            "scroll_segment",
        ]
    )[["session_time"]]
    .agg(
        [
            "count",
            "median",
            "mean",
        ]
    )
    .reset_index()
)

weekly_slot_summary.head(20)


In [ ]:
### 주차별 Session Time 집계

weekly_summary = (
    final_df.groupby(
        [
            "Week",
            "curation_type",
            "app_type",
            "scroll_segment",
        ]
    )[["session_time"]]
    .agg(
        [
            "count",
            "median",
            "mean",
        ]
    )
    .reset_index()
)

weekly_summary.head(20)


In [ ]:
### 후속 콘텐츠 단위 집계 구조 재현

raw = sample.copy()

raw["scroll_segment"] = raw["scroll_segment"].replace(
    "OUT",
    "즉시 이탈",
)

raw["scroll_segment"] = raw["scroll_segment"].replace(
    "0-25%",
    "0% ~ 25%",
)

raw["scroll_segment"] = raw["scroll_segment"].replace(
    "25-50%",
    "25% ~ 50%",
)

raw["scroll_segment"] = raw["scroll_segment"].replace(
    "50-75%",
    "50% ~ 75%",
)

raw["scroll_segment"] = raw["scroll_segment"].replace(
    "75-100%",
    "75% ~ 100%",
)

raw["scroll_segment"] = raw["scroll_segment"].replace(
    "COMPLETE",
    "완독",
)


### 공개용 콘텐츠명 구성

raw["Title"] = raw["content_name"]


### 콘텐츠 유형 및 스크롤 뎁스별 Session Time 집계

raw_summary = (
    raw.groupby(
        [
            "content_type",
            "Title",
            "scroll_segment",
        ]
    )["session_time"]
    .agg(
        [
            "count",
            "mean",
            "median",
        ]
    )
    .reset_index()
)

raw_summary.head(20)


## 체류시간 이상치 제거 기준 설정


In [ ]:
### 이상치 분석 데이터 구성

df = sample[
    [
        "content_id",
        "set_id",
        "user_id",
        "scroll_segment",
        "session_time",
        "content_length",
        "content_type",
        "content_name",
    ]
].copy()

df.replace(
    {
        "scroll_segment": {
            "OUT": 0,
        }
    },
    inplace=True,
)

df.head()


In [ ]:
### 스크롤 뎁스별 Session Time Boxplot 확인

plt.figure(figsize=(12, 6))

sns.boxplot(
    x="scroll_segment",
    y="session_time",
    data=df,
)

plt.xticks(rotation=45)
plt.show()


In [ ]:
### 극단값 범위 제한 후 Boxplot 확인

plt.figure(figsize=(12, 6))

sns.boxplot(
    x="scroll_segment",
    y="session_time",
    data=df[df["session_time"] < 10000],
)

plt.xticks(rotation=45)
plt.show()


In [ ]:
### 300초 초과 Session 분포 확인

plt.figure(figsize=(12, 6))

sns.kdeplot(
    data=df[df["session_time"] > 300],
    x="session_time",
    hue="scroll_segment",
    fill=True,
    common_norm=False,
)

plt.xlim(
    300,
    df["session_time"].max(),
)

plt.show()


In [ ]:
### 300초 이하 Session 데이터 구성

df_300 = df[
    df["session_time"] <= 300
].copy()


### 300초 이하 Session 분포 확인

plt.figure(figsize=(12, 6))

sns.kdeplot(
    data=df_300,
    x="session_time",
    hue="scroll_segment",
    fill=True,
    common_norm=False,
)

plt.xlim(0, 300)
plt.show()


In [ ]:
### 5초 단위 Session Time 구간 생성

last_bin = max(
    300,
    int(np.ceil(df_300["session_time"].max())),
)

bins = list(
    range(
        0,
        last_bin + 5,
        5,
    )
)

labels = [
    f"{i}~{i+5}s"
    for i in bins[:-1]
]

df_300["sessiontime_seg"] = pd.cut(
    df_300["session_time"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

df_300[
    [
        "session_time",
        "sessiontime_seg",
    ]
].head()


In [ ]:
### 콘텐츠 길이 및 스크롤 뎁스별 Session Time 집계

length_summary = (
    df_300.groupby(
        [
            "content_length",
            "scroll_segment",
        ],
        as_index=False,
    )["session_time"]
    .agg(
        [
            "count",
            "mean",
        ]
    )
    .reset_index()
)

length_summary.head(20)


In [ ]:
### 리스트형 콘텐츠 별도 확인

list_type = df_300[
    df_300["content_type"]
    .astype(str)
    .str.contains(
        "List",
        na=False,
    )
].copy()

list_type.head()


In [ ]:
### 100초 이하 Session 데이터 구성

df = df[
    df["session_time"] <= 100
].copy()


### Session Time 구간 크기 비교 함수

def analyze_bin_sizes(
    df,
    bin_sizes=[3, 5, 10],
    max_cutoff=100,
):
    result = []

    for bin_size in bin_sizes:

        last_bin = max(
            max_cutoff,
            int(
                np.ceil(
                    df["session_time"].max()
                )
            ),
        )

        bins = list(
            range(
                0,
                last_bin + bin_size,
                bin_size,
            )
        )

        labels = [
            f"{i}~{i+bin_size}s"
            for i in bins[:-1]
        ]

        temp_df = df.copy()

        temp_df["sessiontime_seg"] = pd.cut(
            temp_df["session_time"],
            bins=bins,
            include_lowest=True,
        )

        temp_df["sessiontime_seg"] = (
            temp_df["sessiontime_seg"]
            .astype(str)
        )

        bin_stat = (
            temp_df.groupby(
                ["sessiontime_seg"]
            )["session_time"]
            .agg(
                [
                    "count",
                    "mean",
                ]
            )
            .reset_index()
        )

        summary = (
            bin_stat.groupby(
                "sessiontime_seg"
            )
            .agg(
                num_bins=(
                    "sessiontime_seg",
                    "nunique",
                ),
                total_usr=(
                    "count",
                    "sum",
                ),
                avg_bin_means=(
                    "mean",
                    "mean",
                ),
                std_bin_mean=(
                    "mean",
                    "std",
                ),
                min_count_per_bin=(
                    "count",
                    "min",
                ),
                max_count_per_bin=(
                    "count",
                    "max",
                ),
            )
            .reset_index()
        )

        summary["bin_size"] = bin_size
        result.append(summary)

    result_df = pd.concat(
        result,
        ignore_index=True,
    )

    return result_df


### 3초, 5초, 10초 단위 구간 비교

result = analyze_bin_sizes(
    df,
    bin_sizes=[3, 5, 10],
    max_cutoff=100,
)

result.head(30)


In [ ]:
### 최종 적용 기준 확인

final_analysis_df = sample[
    sample["session_time"] <= 300
].copy()

print(
    "300초 초과 Session 제외 건수:",
    len(sample) - len(final_analysis_df),
)

print(
    "300초 초과 Session 제외 비율:",
    round(
        (len(sample) - len(final_analysis_df))
        / len(sample)
        * 100,
        2,
    ),
    "%",
)
